In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

# Mayúsculas, sin tildes y sin espacios sobrantes.
def normalizar_texto(valor):
    if pd.isna(valor) or str(valor).strip() in ['', 'nan', 'None']:
        return None
    texto = unicodedata.normalize('NFD', str(valor).strip().upper())
    return ''.join(letra for letra in texto if unicodedata.category(letra) != 'Mn')


In [2]:
# El CSV original debe estar junto al notebook o en la carpeta data del repo.
ruta = Path('activaciones.csv')
if not ruta.exists():
    ruta = Path('../data/activaciones.csv')
activaciones = pd.read_csv(ruta)
activaciones.head()


,activacion_id,año,mes,hora_solicitud,hora_intervencion,tipo_emergencia,distrito,hospital,unidad_id,prioridad,paciente_edad,paciente_sexo
0,1,2024,ENERO,7:10:00,7:17:00,Intoxicación etílica,FUENCARRAL,NaN,39,alta,69.0,NaN
1,2,2024,ENERO,0:17:12,NaN,Agresión sin especificar,CENTRO,NaN,29,baja,65.0,H
2,3,2024,ENERO,0:21:25,0:30:45,Apertura de puerta,LATINA,NaN,15,alta,62.0,H
3,4,2024,ENERO,0:47:26,NaN,Patología cardiovascular,CENTRO,NaN,43,media,75.0,H
4,5,2024,ENERO,0:41:25,0:46:56,Intoxicación etílica,CIUDAD LINEAL,NaN,8,media,50.0,M


In [3]:
activaciones_limpio = activaciones.copy()

## Revisión inicial
Revisamos columnas, nulos, duplicados y valores distintos antes de limpiar. Así vemos los problemas que hay y podemos comprobar después qué ha cambiado.


In [4]:
# 1. ESTRUCTURA GENERAL, NULOS Y DUPLICADOS
print("\n===== 1. ANALISIS GENERAL, NULL, DUPLICADOS =====")
resumen_estructura = pd.DataFrame({
    'Tipo': activaciones_limpio.dtypes,
    'No-Nulos': activaciones_limpio.notnull().sum(),
    'Nulos': activaciones_limpio.isnull().sum(),
    '% Nulos': (activaciones_limpio.isnull().sum() / len(activaciones_limpio) * 100).round(2)
})
print(resumen_estructura)
print(f"\nFilas duplicadas exactas: {activaciones_limpio.duplicated().sum()}")


# 2. ANÁLISIS DE COLUMNAS TEMPORALES (AÑO Y MES)
print("\n===== 2. ANÁLISIS DE COLUMNAS AÑO Y MES =====")
print(f"--- Columna 'año' (Nulos: {activaciones_limpio['año'].isnull().sum()}) ---")
print("Valores únicos detectados en 'año':", activaciones_limpio['año'].unique().tolist())

val_mes_raw = activaciones_limpio['mes'].dropna().astype(str).tolist()
con_tilde_mes = sum(1 for x in val_mes_raw if re.search(r'[áéíóúÁÉÍÓÚ]', x))
con_espacios_mes = sum(1 for x in val_mes_raw if x != x.strip())
con_minusculas_mes = sum(1 for x in val_mes_raw if not x.isupper())

print(f"\n--- Columna 'mes' (Nulos: {activaciones_limpio['mes'].isnull().sum()}) ---")
print("Valores únicos detectados en 'mes':", activaciones_limpio['mes'].unique().tolist())
print("Inconsistencias detectadas en 'mes':")
print(f"  - Con minúsculas / mezcla de caja: {con_minusculas_mes}")
print(f"  - Con tildes: {con_tilde_mes}")
print(f"  - Con espacios sobrantes: {con_espacios_mes}")


# 3. ANÁLISIS DE FORMATO EN HORAS
print("\n===== 3. COLUMNAS DE HORA =====")
for col_hora in ['hora_solicitud', 'hora_intervencion']:
    horas_str = activaciones_limpio[col_hora].dropna().astype(str)
    con_guion = horas_str.str.contains('-').sum()
    con_dos_puntos = horas_str.str.contains(':').sum()
    patron_hh_mm_ss = r'^\d{1,2}:\d{2}:\d{2}$'
    no_cumplen_formato = (~horas_str.str.match(patron_hh_mm_ss)).sum()
    
    print(f"--- Columna '{col_hora}' (Nulos: {activaciones_limpio[col_hora].isnull().sum()}) ---")
    print(f"  - Registros con guión ('-') en vez de ':' : {con_guion}")
    print(f"  - Registros con ':' estándar: {con_dos_puntos}")
    print(f"  - Formato no estándar (ej. 0-03-49): {no_cumplen_formato}")


# 4. RANGOS Y VALORES EN EDAD Y UNIDAD
print("\n===== 4. COHERENCIA DE VARIABLES NUMÉRICAS Y RANGOS =====")
columnas_numericas = ['paciente_edad', 'unidad_id']
print(activaciones_limpio[columnas_numericas].describe().T[['min', 'mean', 'max']])

edades_raras = activaciones_limpio[(activaciones_limpio['paciente_edad'] < 0) | (activaciones_limpio['paciente_edad'] > 115)]
print(f"\n- Pacientes con edad fuera de rango (0-115 años): {len(edades_raras)}")


# 5. VALORES ÚNICOS Y VARIACIONES EN CATEGÓRICAS CRÍTICAS
print("\n===== 5. VALORES ÚNICOS EN VARIABLES CATEGÓRICAS =====")
cols_categoricas = ['tipo_emergencia', 'distrito', 'hospital', 'prioridad', 'paciente_sexo']

for col in cols_categoricas:
    val_raw = activaciones_limpio[col].dropna().astype(str).tolist()
    
    con_tilde = sum(1 for x in val_raw if re.search(r'[áéíóúÁÉÍÓÚ]', x))
    con_espacios = sum(1 for x in val_raw if x != x.strip())
    con_minusculas = sum(1 for x in val_raw if not x.isupper())
    
    print(f"\n--- Columna: '{col}' (Nulos: {activaciones_limpio[col].isnull().sum()} | {(activaciones_limpio[col].isnull().sum()/len(activaciones_limpio)*100):.2f}%) ---")
    print(f"Valores únicos ({activaciones_limpio[col].nunique()} distintos): {activaciones_limpio[col].unique().tolist()[:10]}...")
    print(f"Inconsistencias detectadas:")
    print(f"  - Con minúsculas / mezcla de caja: {con_minusculas}")
    print(f"  - Con tildes: {con_tilde}")
    print(f"  - Con espacios sobrantes: {con_espacios}")


===== 1. ANALISIS GENERAL, NULL, DUPLICADOS =====
                      Tipo  No-Nulos   Nulos  % Nulos
activacion_id        int64    152844       0     0.00
año                  int64    152844       0     0.00
mes                 object    152844       0     0.00
hora_solicitud      object    151258    1586     1.04
hora_intervencion   object    138807   14037     9.18
tipo_emergencia     object    152256     588     0.38
distrito            object    152004     840     0.55
hospital            object     52559  100285    65.61
unidad_id            int64    152844       0     0.00
prioridad           object    152250     594     0.39
paciente_edad      float64    152044     800     0.52
paciente_sexo       object    146810    6034     3.95



Filas duplicadas exactas: 614

===== 2. ANÁLISIS DE COLUMNAS AÑO Y MES =====
--- Columna 'año' (Nulos: 0) ---
Valores únicos detectados en 'año': [2024]

--- Columna 'mes' (Nulos: 0) ---
Valores únicos detectados en 'mes': ['ENERO', 'January', 'enero', 'Enero', '01', 'ENE', 'FEBRERO', 'Febrero', '02', 'February', 'febrero', 'FEB', 'MARZO', '03', 'MAR', 'Marzo', 'marzo', 'March', 'ABRIL', 'ABR', '04', 'April', 'Abril', 'abril', 'MAYO', '05', 'May', 'mayo', 'Mayo', 'MAY', 'JUNIO', 'junio', 'Junio', 'JUN', 'June', '06', 'JULIO', '07', 'Julio', 'July', 'julio', 'JUL', 'AGOSTO', 'Agosto', 'AGO', '08', 'August', 'agosto', 'SEPTIEMBRE', 'SEP', 'septiembre', 'Septiembre', '09', 'September', 'OCTUBRE', '10', 'Octubre', 'octubre', 'October', 'OCT', 'NOVIEMBRE', 'NOV', 'Noviembre', 'November', 'noviembre', '11', 'DICIEMBRE', '12', 'diciembre', 'Diciembre', 'DIC', 'December']
Inconsistencias detectadas en 'mes':
  - Con minúsculas / mezcla de caja: 1696
  - Con tildes: 0
  - Con espacios sobrante

--- Columna 'hora_solicitud' (Nulos: 1586) ---
  - Registros con guión ('-') en vez de ':' : 200
  - Registros con ':' estándar: 150860
  - Formato no estándar (ej. 0-03-49): 398
--- Columna 'hora_intervencion' (Nulos: 14037) ---
  - Registros con guión ('-') en vez de ':' : 0
  - Registros con ':' estándar: 138807
  - Formato no estándar (ej. 0-03-49): 0

===== 4. COHERENCIA DE VARIABLES NUMÉRICAS Y RANGOS =====
                min       mean    max
paciente_edad -10.0  49.559943  199.0
unidad_id       1.0  25.502224   50.0

- Pacientes con edad fuera de rango (0-115 años): 150

===== 5. VALORES ÚNICOS EN VARIABLES CATEGÓRICAS =====



--- Columna: 'tipo_emergencia' (Nulos: 588 | 0.38%) ---
Valores únicos (112 distintos): ['Intoxicación etílica', 'Agresión sin especificar', 'Apertura de puerta', 'Patología cardiovascular', 'Heridas', 'Patología anafiláctica', 'Convulsión y/o epilepsia', 'Patología respiratoria', 'Incendio', nan]...
Inconsistencias detectadas:
  - Con minúsculas / mezcla de caja: 150835
  - Con tildes: 104211
  - Con espacios sobrantes: 0

--- Columna: 'distrito' (Nulos: 840 | 0.55%) ---
Valores únicos (88 distintos): ['FUENCARRAL', 'CENTRO', 'LATINA', 'CIUDAD LINEAL', 'CHAMBERI', 'SAN BLAS', 'RETIRO', 'TETUÁN', 'MONCLOA', 'VALLECAS PTE.']...
Inconsistencias detectadas:
  - Con minúsculas / mezcla de caja: 1175
  - Con tildes: 156
  - Con espacios sobrantes: 0

--- Columna: 'hospital' (Nulos: 100285 | 65.61%) ---
Valores únicos (35 distintos): [nan, 'HULP', 'Clínico San Carlos', 'Concepción (Fund. J. Díaz)', 'HGUGM', 'Gregorio Marañón', 'H. Clínico', 'Ramon Cajal', 'La Princesa', 'La Paz']...
Inconsi


--- Columna: 'prioridad' (Nulos: 594 | 0.39%) ---
Valores únicos (14 distintos): ['alta', 'baja', 'media', '1', 'Media', 'urgente', nan, 'normal', 'BAJA', 'MEDIA']...
Inconsistencias detectadas:
  - Con minúsculas / mezcla de caja: 152034
  - Con tildes: 0
  - Con espacios sobrantes: 0

--- Columna: 'paciente_sexo' (Nulos: 6034 | 3.95%) ---
Valores únicos (2 distintos): [nan, 'H', 'M']...
Inconsistencias detectadas:
  - Con minúsculas / mezcla de caja: 0
  - Con tildes: 0
  - Con espacios sobrantes: 0


## Qué vamos a corregir
Hay meses escritos de varias formas, horas con distintos separadores, edades fuera de rango y nombres con abreviaturas. Unificamos esas formas sin inventar los datos que faltan. Al final quitamos solo las filas que sean iguales en todas las columnas.


## Meses
Dejamos los meses en español y en mayúsculas para no contar el mismo mes como varias categorías. Primero quitamos diferencias de mayúsculas y tildes; después reconocemos números, abreviaturas y nombres en español o inglés. Si aparece una forma no reconocida, se muestra para revisarla.


In [5]:
meses = ['ENERO', 'FEBRERO', 'MARZO', 'ABRIL', 'MAYO', 'JUNIO',
         'JULIO', 'AGOSTO', 'SEPTIEMBRE', 'OCTUBRE', 'NOVIEMBRE', 'DICIEMBRE']
patrones_mes = [r'ENE.*|JAN.*', r'FEB.*', r'MAR.*', r'ABR.*|APR.*',
                r'MAY.*', r'JUN.*', r'JUL.*', r'AGO.*|AUG.*',
                r'SEP.*', r'OCT.*', r'NOV.*', r'DIC.*|DEC.*']

def limpiar_mes(valor):
    texto = normalizar_texto(valor)
    if texto is None:
        return None
    if texto.isdigit() and 1 <= int(texto) <= 12:
        return meses[int(texto) - 1]
    for mes, patron in zip(meses, patrones_mes):
        if re.fullmatch(patron, texto):
            return mes
    return None

activaciones_limpio['mes'] = activaciones['mes'].apply(limpiar_mes)
print('Meses no reconocidos:', activaciones.loc[activaciones_limpio['mes'].isna(), 'mes'].unique())
print('Meses finales:', activaciones_limpio['mes'].unique())


Meses no reconocidos: []
Meses finales: ['ENERO' 'FEBRERO' 'MARZO' 'ABRIL' 'MAYO' 'JUNIO' 'JULIO' 'AGOSTO'
 'SEPTIEMBRE' 'OCTUBRE' 'NOVIEMBRE' 'DICIEMBRE']


## Horas
Cambiamos puntos y guiones por dos puntos y añadimos ceros a la izquierda para dejar HH:MM:SS. No rellenamos las horas que faltan porque no tenemos información para saberlas. Comprobamos también que horas, minutos y segundos estén dentro de su rango.


In [6]:
def estandarizar_hora_completa(val):
    if pd.isna(val) or str(val).strip() in ['', 'nan', 'None']:
        return None
    
    # 1. Limpieza de texto y reemplazo de puntos y guiones por dos puntos
    txt = str(val).strip().replace('-', ':').replace('.', ':')
    
    # 2. Dividir componentes (HH:MM:SS)
    partes = txt.split(':')
    
    try:
        if len(partes) == 3:
            h, m, s = partes
            return f"{int(h):02d}:{int(m):02d}:{int(s):02d}"
        elif len(partes) == 2:
            h, m = partes
            return f"{int(h):02d}:{int(m):02d}:00"
    except ValueError:
        return txt
    
    return txt

# Aplicar a ambas columnas
activaciones_limpio['hora_solicitud'] = activaciones_limpio['hora_solicitud'].apply(estandarizar_hora_completa)
activaciones_limpio['hora_intervencion'] = activaciones_limpio['hora_intervencion'].apply(estandarizar_hora_completa)

# VERIFICACIÓN DEFINITIVA
print("===== VERIFICACIÓN DEFINITIVA: FORMATO DE HORAS =====")
patron_estandar = r'^\d{2}:\d{2}:\d{2}$'

for col in ['hora_solicitud', 'hora_intervencion']:
    horas_str = activaciones_limpio[col].dropna()
    cumplen_patron = horas_str.str.match(patron_estandar).sum()
    
    print(f"--- Columna '{col}' ---")
    print(f"• Total no nulos: {len(horas_str)}")
    print(f"• Cumplen formato estricto HH:MM:SS: {cumplen_patron} de {len(horas_str)}")
    print(f"• Nulos mantenidos intactos: {activaciones_limpio[col].isna().sum()}")
for columna in ['hora_solicitud', 'hora_intervencion']:
    horas = pd.to_timedelta(activaciones_limpio[columna], errors='coerce')
    no_validas = activaciones_limpio[columna].notna() & (horas.isna() | (horas < pd.Timedelta(0)) | (horas >= pd.Timedelta(days=1)))
    print(columna, '- horas fuera de rango:', no_validas.sum())
    activaciones_limpio.loc[no_validas, columna] = None


===== VERIFICACIÓN DEFINITIVA: FORMATO DE HORAS =====
--- Columna 'hora_solicitud' ---
• Total no nulos: 151258
• Cumplen formato estricto HH:MM:SS: 151258 de 151258
• Nulos mantenidos intactos: 1586
--- Columna 'hora_intervencion' ---
• Total no nulos: 138807
• Cumplen formato estricto HH:MM:SS: 138807 de 138807
• Nulos mantenidos intactos: 14037


hora_solicitud - horas fuera de rango: 0


hora_intervencion - horas fuera de rango: 0


## Edad
Conservamos edades entre 0 y 115 años. Es un rango razonable que se ha aceptado para el proyecto. Las edades negativas o mayores de 115 pasan a nulo; no ponemos una edad media porque sería inventar la edad de esa persona. Las filas se conservan.


In [7]:
# Convertir a NaN los valores de paciente_edad menores a 0 o mayores a 115
activaciones_limpio['paciente_edad'] = activaciones_limpio['paciente_edad'].apply(
    lambda x: np.nan if (pd.isna(x) or x < 0 or x > 115) else x
)

# VERIFICACIÓN DEFINITIVA DE EDAD
print("===== VERIFICACIÓN: COLUMNA PACIENTE_EDAD =====")
print("• Mínimo de edad:", activaciones_limpio['paciente_edad'].min())
print("• Máximo de edad:", activaciones_limpio['paciente_edad'].max())
print("\n• Total de valores nulos (NaN) en 'paciente_edad':", activaciones_limpio['paciente_edad'].isna().sum())
print("• Conteo de registros fuera del rango [0, 115]:", 
      ((activaciones_limpio['paciente_edad'] < 0) | (activaciones_limpio['paciente_edad'] > 115)).sum())

===== VERIFICACIÓN: COLUMNA PACIENTE_EDAD =====
• Mínimo de edad: 0.0
• Máximo de edad: 99.0

• Total de valores nulos (NaN) en 'paciente_edad': 950
• Conteo de registros fuera del rango [0, 115]: 0


## Distrito, hospital y prioridad
Quitamos tildes y diferencias de mayúsculas para juntar etiquetas que significan lo mismo. Aplicamos reglas de texto a las abreviaturas y errores que hemos encontrado, sin asignar a la fuerza un valor desconocido.

Las formas VALLECAS PTE., P. VALLECAS y PUENTE DE VALLECAS son Puente de Vallecas. VILLA DE VALLECAS se queda en Villa. VALLECAS a secas pasa a NO CONSTA porque no permite distinguir los dos distritos (más abajo, en "Vallecas a secas", se explica qué otros datos hemos probado para decidirlo y por qué no sirven). Esas filas se conservan para demanda, pero no se usan al comparar distritos. Los registros de fuera del municipio también se mantienen en esta limpieza.

La prioridad queda en ALTA, MEDIA o BAJA: 1 y URGENTE son alta, 2 y NORMAL son media y 3 es baja. Esta equivalencia está confirmada para el proyecto.

Unificamos las abreviaturas de hospitales y usamos HOSPITAL 12 DE OCTUBRE para DOCE DE OCTUBRE. Si un nombre no encaja en una regla, se conserva y se muestra para revisarlo.


In [8]:
#primero visualizamos todas las distintas etiquetas existentes
cols_a_inspeccionar = ['distrito', 'hospital', 'prioridad']

for col in cols_a_inspeccionar:
    print(f"===== VALORES ÚNICOS EN '{col}' (Total Nulos: {activaciones_limpio[col].isna().sum()}) =====")
    # Obtener valores únicos convertidos a lista
    unicos = activaciones_limpio[col].dropna().unique().tolist()
    print(unicos)
    print("\n")

===== VALORES ÚNICOS EN 'distrito' (Total Nulos: 840) =====
['FUENCARRAL', 'CENTRO', 'LATINA', 'CIUDAD LINEAL', 'CHAMBERI', 'SAN BLAS', 'RETIRO', 'TETUÁN', 'MONCLOA', 'VALLECAS PTE.', 'VICALVARO', 'ARGANZUELA', 'SALAMANCA', 'USERA', 'TETUAN', 'CARABANCHEL', 'MORATALAZ', 'HORTALEZA', 'VILLA DE VALLECAS', 'CHAMBERY', 'CHAMARTIN', 'VILLAVERDE', 'ARGANZELA', 'BARAJAS', 'Vallecas', 'LATNA', 'S. BLAS', 'barajas', 'P. VALLECAS', 'CENTO', 'TETAN', 'villaverde', 'CTR', 'arganzuela', 'FUERA TERMINO MUNICIPAL', 'Moncloa', 'USRA', 'Arganzuela', 'centro', 'Centro', 'usera', 'CDAD LINEAL', 'fuencarral', 'vicalvaro', 'Carabanchel', 'Tetuán', 'C. LINEAL', 'MONCLOA-ARAVACA', 'CRABANCHEL', 'Latina', 'San Blas', 'HORTLEZA', 'Hortaleza', 'PUENTE DE VALLECAS', 'Retiro', 'chamberi', 'FUENCARRAL-EL PARDO', 'tetuan', 'Ciudad Lineal', 'Barajas', 'CHAMBERÍ', 'Chamberí', 'salamanca', 'Salamanca', 'Usera', 'moncloa', 'retiro', 'Moratalaz', 'Villaverde', 'MORATALA', 'Fuencarral', 'carabanchel', 'SAN BLAS-CANILLEJA

In [9]:
reglas_distrito = [
    ('ARGANZELA', 'ARGANZUELA'),
    ('BARJAS', 'BARAJAS'),
    ('CRABANCHEL', 'CARABANCHEL'),
    ('CENTO|CTR', 'CENTRO'),
    ('CHAMBERY', 'CHAMBERI'),
    ('CDAD\\ LINEAL|C\\.\\ LINEAL', 'CIUDAD LINEAL'),
    ('FUENCARRAL', 'FUENCARRAL-EL PARDO'),
    ('HORTLEZA', 'HORTALEZA'),
    ('LATNA|LA\\ LATINA', 'LATINA'),
    ('MONCLOA', 'MONCLOA-ARAVACA'),
    ('MORATALA', 'MORATALAZ'),
    ('VALLECAS\\ PTE\\.|P\\.\\ VALLECAS', 'PUENTE DE VALLECAS'),
    ('VALLECAS', 'NO CONSTA'),
    ('EL\\ RETIRO|RTIRO', 'RETIRO'),
    ('SALMANCA', 'SALAMANCA'),
    ('SAN\\ BLAS|S\\.\\ BLAS', 'SAN BLAS-CANILLEJAS'),
    ('TETAN', 'TETUAN'),
    ('USRA', 'USERA'),
    ('VILLA\\ VERDE', 'VILLAVERDE'),
    ('FUERA\\ TERMINO\\ MUNICIPAL|POZUELO|C\\.A\\.M\\.|LEGANES', 'FUERA DE TERMINO'),
]

reglas_hospital = [
    ('HULP|LA\\ PAZ|H\\.\\ LA\\ PAZ', 'HOSPITAL LA PAZ'),
    ('CLINICO\\ SAN\\ CARLOS|H\\.\\ CLINICO|SAN\\ CARLOS|HCSC', 'HOSPITAL CLINICO SAN CARLOS'),
    ('CONCEPCION\\ \\(FUND\\.\\ J\\.\\ DIAZ\\)', 'HOSPITAL FUNDACION JIMENEZ DIAZ'),
    ('HGUGM|GREGORIO\\ MARANON|MARANON|H\\.\\ GREGORIO\\ MARANON', 'HOSPITAL GREGORIO MARANON'),
    ('RAMON\\ CAJAL|RAMON\\ Y\\ CAJAL|RYC|H\\.\\ RAMON\\ Y\\ CAJAL', 'HOSPITAL RAMON Y CAJAL'),
    ('LA\\ PRINCESA', 'HOSPITAL LA PRINCESA'),
    ('DOCE\\ DE\\ OCTUBRE', 'HOSPITAL 12 DE OCTUBRE'),
    ('INFANTA\\ LEONOR', 'HOSPITAL INFANTA LEONOR'),
    ('CENTRAL\\ DE\\ LA\\ DEFENSA', 'HOSPITAL CENTRAL DE LA DEFENSA GOMEZ ULLA'),
    ('NINO\\ JESUS\\ \\(INFANTIL\\)', 'HOSPITAL NINO JESUS'),
    ('MONCLOA\\ \\(ASISA\\)', 'HOSPITAL MONCLOA'),
    ('PUERTA\\ DE\\ HIERRO', 'HOSPITAL PUERTA DE HIERRO'),
    ('ALCALA\\ DE\\ HENARES\\ \\(PPE\\.\\ DE\\ ASTURIAS\\)', 'HOSPITAL PRINCIPE DE ASTURIAS'),
    ('GETAFE', 'HOSPITAL DE GETAFE'),
    ('SANTA\\ CRISTINA', 'HOSPITAL SANTA CRISTINA'),
    ('DOCTOR\\ RODRIGUEZ\\ LAFORA', 'HOSPITAL DOCTOR RODRIGUEZ LAFORA'),
    ('MILITARES\\ \\(AIRE,POLIC\\.\\ NAVAL,\\ OTROS\\.\\.\\.\\)', 'CENTROS MILITARES'),
    ('CENTROS\\ MUNICIPALES\\ \\(ASIST\\.\\ Y\\ \\ ALBERGUES\\)|CENTROS\\ MUNICIPALES\\ \\(ASIST\\.\\ Y\\ ALBERGUES\\)', 'CENTROS MUNICIPALES'),
]

def aplicar_reglas(valor, reglas):
    texto = normalizar_texto(valor)
    if texto is None:
        return None
    for patron, resultado in reglas:
        if re.fullmatch(patron, texto):
            return resultado
    return texto

for columna, reglas in [('distrito', reglas_distrito), ('hospital', reglas_hospital)]:
    activaciones_limpio[columna] = activaciones[columna].apply(lambda valor: aplicar_reglas(valor, reglas))
    conocidos = {resultado for _, resultado in reglas}
    sin_regla = activaciones_limpio[columna].dropna()
    print(columna, '- valores conservados sin sustitución:', sorted(set(sin_regla) - conocidos))

def limpiar_prioridad(valor):
    texto = normalizar_texto(valor)
    if texto in ['1', 'URGENTE', 'ALTA']:
        return 'ALTA'
    if texto in ['2', 'NORMAL', 'MEDIA']:
        return 'MEDIA'
    if texto in ['3', 'BAJA']:
        return 'BAJA'
    return None

activaciones_limpio['prioridad'] = activaciones['prioridad'].apply(limpiar_prioridad)
print('Prioridades no reconocidas:', activaciones.loc[activaciones_limpio['prioridad'].isna(), 'prioridad'].unique())
print(activaciones_limpio['prioridad'].value_counts(dropna=False))


distrito - valores conservados sin sustitución: ['CARRETERAS Y CIRCUNVALACIONES', 'CHAMARTIN', 'VICALVARO', 'VILLA DE VALLECAS']


hospital - valores conservados sin sustitución: ['OTROS CENTROS PUBLICOS']


Prioridades no reconocidas: [nan]
prioridad
MEDIA    76109
ALTA     45442
BAJA     30699
None       594
Name: count, dtype: int64


## Tipo de emergencia
Pasamos los textos a mayúsculas y quitamos tildes y espacios de los extremos. Así no quedan tipos separados solo por cómo están escritos. Aquí no fusionamos tipos por significado ni inventamos protocolos: eso necesita una revisión aparte del catálogo.


In [10]:
# Función para quitar tildes y pasar a mayúsculas
def limpiar_emergencia(val):
    if pd.isna(val) or str(val).strip() in ['', 'nan', 'None']:
        return None
    
    texto = str(val).strip().upper()
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto) 
        if unicodedata.category(c) != 'Mn'
    )
    return texto

# Aplicar a tipo_emergencia
activaciones_limpio['tipo_emergencia'] = activaciones_limpio['tipo_emergencia'].apply(limpiar_emergencia)

# VERIFICACIÓN
print("===== VERIFICACIÓN: TIPO_EMERGENCIA =====")
print("• Total de nulos en 'tipo_emergencia':", activaciones_limpio['tipo_emergencia'].isna().sum())
print("• Número de categorías únicas tras normalizar:", activaciones_limpio['tipo_emergencia'].nunique())
print("\nPrimeras 15 categorías resultantes:")
print(activaciones_limpio['tipo_emergencia'].dropna().unique()[:15])

===== VERIFICACIÓN: TIPO_EMERGENCIA =====
• Total de nulos en 'tipo_emergencia': 588
• Número de categorías únicas tras normalizar: 103

Primeras 15 categorías resultantes:
['INTOXICACION ETILICA' 'AGRESION SIN ESPECIFICAR' 'APERTURA DE PUERTA'
 'PATOLOGIA CARDIOVASCULAR' 'HERIDAS' 'PATOLOGIA ANAFILACTICA'
 'CONVULSION Y/O EPILEPSIA' 'PATOLOGIA RESPIRATORIA' 'INCENDIO'
 'CERTIFICACION PSIQUIATRICO' 'CASUAL: CAIDA, ETC'
 'ACCIDENTE MENOS DE 3 VICTIMAS' 'OTROS' 'ORDEN MEDICA'
 'PATOLOGIA PSICOSOMATICA']


## Datos que faltan
En las columnas de texto dejamos NO CONSTA cuando falta el dato. En hospital, un vacío no demuestra que no hubo traslado, así que no usamos SIN TRASLADO. Las horas y edades que faltan siguen como nulos. No añadimos el día porque no existe en la fuente y no se puede recuperar con estos datos.


In [11]:
# 1. Imputación en 'hospital'
activaciones_limpio['hospital'] = activaciones_limpio['hospital'].fillna('NO CONSTA')

# 2. Imputación en 'paciente_sexo'
activaciones_limpio['paciente_sexo'] = activaciones_limpio['paciente_sexo'].fillna('NO CONSTA')

# 3. Imputación en 'tipo_emergencia' y 'distrito' (si se desea eliminar nulos por completo en categóricas)
activaciones_limpio['tipo_emergencia'] = activaciones_limpio['tipo_emergencia'].fillna('NO CONSTA')
activaciones_limpio['distrito'] = activaciones_limpio['distrito'].fillna('NO CONSTA')
activaciones_limpio['prioridad'] = activaciones_limpio['prioridad'].fillna('NO CONSTA')

# VERIFICACIÓN DE RELLENO
print("===== VERIFICACIÓN TRAS IMPUTACIÓN DE CATEGÓRICAS =====")
print(activaciones_limpio[['hospital', 'paciente_sexo', 'tipo_emergencia', 'distrito', 'prioridad']].isna().sum())

===== VERIFICACIÓN TRAS IMPUTACIÓN DE CATEGÓRICAS =====
hospital           0
paciente_sexo      0
tipo_emergencia    0
distrito           0
prioridad          0
dtype: int64


## Vallecas a secas

En la limpieza de distritos, las filas que solo dicen VALLECAS pasan a NO CONSTA porque no se sabe si son Puente o Villa de Vallecas. Bluetab nos dijo por correo que ese es el único valor ambiguo (39 filas) y que podíamos excluirlas del análisis por distrito o asignarlas al distrito más frecuente, documentándolo. En la reunión del 15/10 añadieron que quizá se podía resolver con otros datasets. Probamos con los que traen alguna pista:

- **unidades_samur** (por `unidad_id`): miramos la base de la unidad que atendió la activación. Si la unidad saliera siempre de la base de su distrito, serviría. Comprobamos si eso pasa en el resto de distritos.
- **hospitales** (columna `hospital`): miramos si Puente y Villa de Vallecas se reparten los hospitales de forma distinta.
- **accidentes, meteorología y calendario**: no tienen `activacion_id` ni `unidad_id` en común con activaciones (accidentes solo trae `unidad_samur_id`, y los otros van por fecha), así que no se pueden cruzar con estas filas.

In [12]:
def leer_csv(nombre):
    ruta = Path(nombre)
    if not ruta.exists():
        ruta = Path('../data') / nombre
    return pd.read_csv(ruta)

unidades = leer_csv('unidades_samur.csv')
unidades['base_distrito'] = unidades['base_distrito'].apply(normalizar_texto)

v = activaciones_limpio.copy()
v['base_unidad'] = v['unidad_id'].map(unidades.set_index('unidad_id')['base_distrito'])

# Grupo de cada fila: VALLECAS a secas (en el original), Puente o Villa
original = activaciones['distrito'].apply(normalizar_texto)
v['grupo'] = None
v.loc[original == 'VALLECAS', 'grupo'] = 'VALLECAS A SECAS'
v.loc[v['distrito'] == 'PUENTE DE VALLECAS', 'grupo'] = 'PUENTE'
v.loc[v['distrito'] == 'VILLA DE VALLECAS', 'grupo'] = 'VILLA'
print(v['grupo'].value_counts().to_string())
print(f"Las {(v['grupo'] == 'VALLECAS A SECAS').sum()} filas son el {(v['grupo'] == 'VALLECAS A SECAS').sum() / len(v) * 100:.3f}% de las {len(v)} filas (antes de quitar duplicados)")
print()

# 1. ¿La unidad sale de la base de su distrito? Lo miramos en los distritos que tienen base.
distritos_con_base = set(unidades['base_distrito'].dropna())
con_base = v[v['base_unidad'].notna() & v['distrito'].isin(distritos_con_base)]
print(f"Unidad con base en el mismo distrito de la activación: {(con_base['base_unidad'] == con_base['distrito']).mean() * 100:.1f}%")
print()
print("Base de la unidad en cada grupo (% por columna):")
tabla_unidad = pd.crosstab(v['base_unidad'].fillna('SIN BASE'), v['grupo'])
print((tabla_unidad / tabla_unidad.sum() * 100).round(1).to_string())
print()

# 2. Hospital
vallecas = v[v['grupo'].notna()]
print("Hospital en cada grupo (nº de filas):")
print(pd.crosstab(vallecas['hospital'], vallecas['grupo']).sort_values('PUENTE', ascending=False).head(6).to_string())
print()
puente = (v['grupo'] == 'PUENTE').sum(); villa = (v['grupo'] == 'VILLA').sum()
print(f"% de Puente sobre Puente + Villa: {puente / (puente + villa) * 100:.1f}%")
infanta = v[v['hospital'] == 'HOSPITAL INFANTA LEONOR']['grupo'].value_counts()
print(f"% de Puente entre las filas de Puente y Villa que van a Infanta Leonor: {infanta['PUENTE'] / (infanta['PUENTE'] + infanta['VILLA']) * 100:.1f}%")

grupo
PUENTE              10035
VILLA                3910
VALLECAS A SECAS       39
Las 39 filas son el 0.026% de las 152844 filas (antes de quitar duplicados)

Unidad con base en el mismo distrito de la activación: 11.0%

Base de la unidad en cada grupo (% por columna):
grupo          PUENTE  VALLECAS A SECAS  VILLA
base_unidad                                   
CARABANCHEL       7.9              10.3    8.3
CENTRO            5.9               5.1    5.3
CIUDAD LINEAL     8.3              12.8    8.5
FUENCARRAL       10.6               5.1   10.2
HORTALEZA         7.8              10.3    7.6
LATINA           19.9              23.1   19.2
SIN BASE          8.1               7.7    8.1
TETUAN           17.7              15.4   18.3
VALLECAS         13.8              10.3   14.7

Hospital en cada grupo (nº de filas):
grupo                      PUENTE  VALLECAS A SECAS  VILLA
hospital                                                  
NO CONSTA                    6585                24   

### Qué hemos decidido

Los otros datasets no sirven para repartir esas filas:

- La unidad que atiende no depende del distrito: en el resto de distritos con base solo coincide la base de la unidad con el distrito de la activación solo en torno al 11% de las veces, que es más o menos una de cada ocho, lo que saldría repartiendo las unidades al azar entre las bases. Y las unidades de las 39 filas salen de varias bases distintas, como las de Puente y Villa.
- El hospital tampoco distingue: solo 15 de las 39 filas tienen hospital (9 Gregorio Marañón y 5 Infanta Leonor) y los dos hospitales reciben activaciones de Puente y de Villa. Gregorio Marañón recibe bastantes más de Puente que de Villa (1.574 frente a 266) e Infanta Leonor reparte 59% Puente y 41% Villa. Eso solo da una probabilidad, no una certeza, y asignar por probabilidad sería inventar el dato.
- Accidentes, meteorología y calendario no comparten ninguna clave con estas filas.

Por eso mantenemos NO CONSTA. Las razones:

1. Asignarlas al distrito más frecuente (Puente) sería inventar un dato: aunque Puente sea el más frecuente, una parte de esas filas seguro que son de Villa.
2. Son muy pocas (0,03% de las filas), así que dejarlas fuera del análisis por distrito casi no cambia los resultados.
3. Siguen contando en la demanda total, porque la activación es válida.

Es una de las dos opciones que propuso Bluetab (excluirlas del análisis por distrito, documentándolo). Las demás variantes (VALLECAS PTE., P. VALLECAS, PUENTE DE VALLECAS) sí se asignan a Puente de Vallecas, como confirmaron.

## Duplicados
Quitamos solo las filas iguales en todas las columnas, después de unificar los textos. Así evitamos contar varias veces el mismo registro. Si un activacion_id se repite pero el resto de datos cambia, conservamos las filas porque no sabemos cuál es la correcta.


In [13]:
n_antes = len(activaciones_limpio)
print(f"Filas duplicadas exactas tras la homogeneización: {activaciones_limpio.duplicated().sum()}")
activaciones_limpio = activaciones_limpio.drop_duplicates().reset_index(drop=True)
print(f"Filas antes: {n_antes} | después: {len(activaciones_limpio)}")
print("activacion_id repetidos con contenido distinto (se conservan):", activaciones_limpio['activacion_id'].duplicated().sum())

Filas duplicadas exactas tras la homogeneización: 737


Filas antes: 152844 | después: 152107
activacion_id repetidos con contenido distinto (se conservan): 63


## Comprobación final y exportación
Mostramos el tamaño final, los nulos y los estadísticos para revisar el resultado. Exportamos las mismas 12 columnas, sin columnas de flags. El original no se modifica.


In [14]:
print('Tamaño final:', activaciones_limpio.shape)
print('Nulos por columna:')
print(activaciones_limpio.isna().sum())
print('Resumen numérico:')
display(activaciones_limpio.describe().round(2))
print('Resumen de columnas de texto:')
display(activaciones_limpio.describe(include='object'))
print('Distrito NO CONSTA:', activaciones_limpio['distrito'].eq('NO CONSTA').sum())
print('Hospital NO CONSTA:', activaciones_limpio['hospital'].eq('NO CONSTA').sum())
print('Hospital 12 de Octubre:', activaciones_limpio['hospital'].eq('HOSPITAL 12 DE OCTUBRE').sum())
print('Duplicados exactos restantes:', activaciones_limpio.duplicated().sum())
assert list(activaciones_limpio.columns) == list(activaciones.columns)
assert activaciones_limpio.shape == (152107, 12)
assert not activaciones_limpio.duplicated().any()
activaciones_limpio.to_csv('activaciones_limpio.csv', index=False)


Tamaño final: (152107, 12)
Nulos por columna:


activacion_id            0
año                      0
mes                      0
hora_solicitud        1579
hora_intervencion    13976
tipo_emergencia          0
distrito                 0
hospital                 0
unidad_id                0
prioridad                0
paciente_edad          950
paciente_sexo            0
dtype: int64
Resumen numérico:


,activacion_id,año,unidad_id,paciente_edad
count,152107.00,152107.0,152107.0,151157.00
mean,78731.40,2024.0,25.5,49.52
std,67579.14,0.0,14.4,28.90
min,1.00,2024.0,1.0,0.00
25%,38010.50,2024.0,13.0,25.00
50%,76023.00,2024.0,25.0,49.00
75%,114036.50,2024.0,38.0,75.00
max,1051430.00,2024.0,50.0,99.00


Resumen de columnas de texto:


,mes,hora_solicitud,hora_intervencion,tipo_emergencia,distrito,hospital,prioridad,paciente_sexo
count,152107,150528,138131,152107,152107,152107,152107,152107
unique,12,66211,63607,104,24,20,4,3
top,JUNIO,17:28:48,18:27:24,PATOLOGIA CARDIOVASCULAR,CENTRO,NO CONSTA,MEDIA,M
freq,13760,76,74,28144,22144,99787,75733,73283


Distrito NO CONSTA: 876
Hospital NO CONSTA: 99787


Hospital 12 de Octubre: 8220


Duplicados exactos restantes: 0


## Pendiente: cruces de medianoche
La fuente solo tiene año, mes y horas. Hay 616 filas con la intervención antes que la solicitud. Bluetab nos contestó por correo (14/10) lo que esperan:

- Asumir que solicitud e intervención son del mismo día, salvo que haya un cruce de medianoche.
- Definir una regla explícita para ese cruce (por ejemplo, solicitud a última hora del día e intervención a primera hora del siguiente: se suman 24 horas).
- Las filas que no encajen en la regla se consideran errores de registro: el tiempo de respuesta queda nulo, pero la activación no se elimina porque sigue valiendo para la demanda.
- Lo que se evalúa es que la regla esté justificada y que se cuente cuántas filas caen en cada caso. Según Bluetab, bien planteada, la gran mayoría de las 616 son cruces de medianoche.

Todavía no hemos fijado la regla concreta (qué horas cuentan como "última hora" y "primera hora"), por eso no calculamos ni exportamos un tiempo de respuesta en este CSV. Tampoco cambiamos las horas ni añadimos un día supuesto. Es el siguiente paso.